# Drug Separation

These are clearly separating much more strongly by cell line than perturbation. here, we will attempt to achieve an embedding that demonstrates good drug perturbation separation.

In [ ]:
import os
import ast
import json
import joblib
from joblib import Parallel, delayed
import time

from tqdm import tqdm
from tqdm import trange

import numpy as np
import pandas as pd

from scipy.stats import f_oneway, kruskal
from scipy import stats
from sklearn.metrics import normalized_mutual_info_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

from kneed import KneeLocator

import scanpy as sc
import umap

import matplotlib.pyplot as plt
import seaborn as sns

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io

In [ ]:
n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
author = 'Tahoe100M'

In [ ]:
tf_adata = io.read_tfad(os.path.join(data_path, 'processed', author + '_consensus_tf_activity.h5ad'))

Step 1: Fit a PLS-DA model of drug ~ TF activity , including confounders (cell line, plate, cell cycle scores) as X covariates to remove those sources of variance.
- since only cell line information will be available in the generative model, we only control for this (we need to be able to project the predicted data into this space)
#- categorical confounders (cell line and plate) are one-hot encoded
#- continuous confounders (cell cycle scores) are scaled
- X (TF activity) is not scaled because it is the consensus Z-score from decoupler

We will identify an optimal n components by assessing the following:
1) Mean accuracy score across 5-fold CV of a logistic regression classifier trained on the PLS components (cannot directly use PLS model CV because it is technically a regression model on the one-hot encodings, and the high R^2 does not necessarily mean high classification accuracy)
2) Explained variance in the X and y blocks

In [ ]:
def ss_explained_var(Y, Y_pred):
    ss_res = np.sum((Y - Y_pred) ** 2)
    ss_tot = np.sum((Y - np.mean(Y, axis=0)) ** 2)
    return 1 - ss_res / ss_tot


def prepare_input_matrix(tf_adata, 
                         control_confounders = True, 
                        enc_X = None):
    
    """Prepares input for PLSR, and fits one-hot encodings of confounding categorical covariates (cell line)"""
    X = tf_adata.X
    if control_confounders:
        # covariates
#         cell_line = tf_adata.obs['cell_line'].astype(str).values
#         plate = tf_adata.obs['plate'].astype(str).values

#         enc_X = OneHotEncoder(sparse_output=False, drop='first')  # drop to avoid collinearity
#         covariates = enc_X.fit_transform(np.stack([cell_line, plate], axis=1)) # plate + cell

#         cell_cycle_scores = np.concatenate([
#             tf_adata.obs['S_score'].values.reshape(-1, 1),
#             tf_adata.obs['G2M_score'].values.reshape(-1, 1)
#         ], axis=1)

#         scaler = StandardScaler()
#         cell_cycle_scaled = scaler.fit_transform(cell_cycle_scores)

#         X = np.concatenate([X, covariates, cell_cycle_scaled], axis=1)

        cell_line = tf_adata.obs['cell_line'].astype(str).values.reshape(-1, 1)
    
        if enc_X is None:
            enc_X = OneHotEncoder(sparse_output=False, drop='first')
            enc_X.fit(cell_line)
        covariates = enc_X.transform(cell_line)

        X = np.concatenate([X, covariates], axis=1)
    return X, enc_X


def pls_da(tf_adata, 
           n_components: int, 
          control_confounders: bool = True, 
          assess: bool = True, 
           return_components: bool = True,
          seed: int = 888, 
          enc_X = None, 
          enc_Y = None):
    """Creates a PLS-DA model (drug ~ TF activity) with confounding covariates

    Parameters
    ----------
    tf_adata : _type_
        TF activity anndata object
    n_components : int
        Number of PLS components to use
    control_confounders : bool, optional
        controls for various confounders in the X-block of PLS-DA, by default True
    assess : bool, optional
        gets assessment metrics for PLS fit, by default True
    return_components : bool, optional
        returns the X PLS components, by default True
    seed : int, optional
        random state, by default 888
    enc_X: 
        fit model for encoding. If not provided (when fitting on actual data) will fit an encoding. 
        If provided (when projecting predicted data), will use the fit encoding to transform the input
    """
    

    y = tf_adata.obs['drug'].astype(str).values.reshape(-1,1)
    if enc_Y is None:
        enc_Y = OneHotEncoder(sparse_output=False, drop=None)    
        enc_Y.fit(y)
    Y = enc_Y.transform(y)

    X, enc_X = prepare_input_matrix(tf_adata = tf_adata,
                                    control_confounders = control_confounders, 
                                   enc_X = enc_X)

    pls_model = PLSRegression(n_components=n_components)
    pls_model.fit(X, Y)
    
    X_pls = None
    if assess or return_components:
        X_pls = pls_model.transform(X)
    
    assessment = None
    if assess: 
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        accuracy_score = cross_val_score(clf, X_pls, y.ravel(), 
                                         cv=StratifiedKFold(5, random_state=seed, shuffle=True), 
                                         scoring='accuracy').mean()
        
#         explained_var_y = np.var(pls_model.y_scores_, axis=0, ddof=0) / np.var(Y, axis=0, ddof=0).sum()
#         

        Y_pred = pls_model.predict(X)
        explained_var_y = ss_explained_var(Y, Y_pred) #1 - np.sum((Y - Y_pred) ** 2) / np.sum((Y - Y.mean(axis=0)) ** 2)
        
        
#         X_pred = pls_model.x_scores_ @ pls_model.x_loadings_.T
#         explained_var_x = ss_explained_var(X, X_pred)
#         explained_var_x = np.var(pls_model.x_scores_, axis=0) / np.var(X, axis=0).sum()
#         cum_explained_x = np.cumsum(explained_var_x)[-1]

        assessment = {
            'n_components': n_components,
            'accuracy': accuracy_score,
            'explained_y': explained_var_y}  
    
    models = {'pls_model': pls_model, 
             'encoder_x': enc_X, 
             'encoder_y': enc_Y}

    return models, assessment, X_pls


In [ ]:
assessment_df = []
pls_components_max = 25
for n_components in trange(1, pls_components_max + 1):
    _, assessment, _= pls_da(tf_adata = tf_adata, 
                                 n_components = n_components ,
                                 control_confounders = True, 
                                 assess = True,
                             return_components = True, 
                                 seed = seed, 
                            enc_X = None, 
                            enc_Y = None)
    assessment_df.append(assessment)
    
assessment_df = pd.DataFrame(assessment_df)
assessments_df.to_csv(os.path.join(data_path, 'interim', author + '_PLS_drug_scores.csv'))

# # run in parallel
# def run_pls(n_components):
#     pls_model, assessment, X_pls = pls_da(tf_adata=tf_adata,
#                                    n_components=n_components,
#                                    control_confounders=True,
#                                    assess=True,
#                                       return_components = True,    
#                                    seed=seed)
#     return pls_model, assessment

# results = Parallel(n_jobs=min(n_cores, pls_components_max))(  
#     delayed(run_pls)(n) for n in tqdm(range(1, pls_components_max + 1))
# )
# models, assessments, _ = zip(*results)
# assessments_df = pd.DataFrame(assessments)


# assessments_df.to_csv(os.path.join(data_path, 'interim', author + '_PLS_drug_scores.csv'))
# best_idx = assessments_df['accuracy'].idxmax()
# joblib.dump(models[best_idx],
#             os.path.join(data_path, 'processed', author + '_PLS_drug_model.joblib'))


# you are here A: run with actual results from here

In [ ]:
res = pd.read_csv(os.path.join(data_path, 'interim', author + '_PLS_drug_scores.csv'), index_col =0)

In [ ]:
fig, ax = plt.subplots(ncols = 2, figsize=(10,5))

for (j, metric) in enumerate(['accuracy', 'explained_y']):
    sns.scatterplot(data = res, x = 'n_components', y = metric, ax = ax[j])
    
fig.tight_layout()
;

In [ ]:
y_ax = res.accuracy
x_ax = np.array(range(len(y_ax))) + 1
kneedle = KneeLocator(x = x_ax, y = y_ax, curve='convex', direction='decreasing')
n_component_elbow = kneedle.elbow
n_component_elbow

For now, we won't do permutation testing to assess the significance of the model since it takes a long time to fit. We will simply assess by the added CV accuracy and variance explained.

Let's see whether the PLS components can better explain the variance when compared to PCA:

In [ ]:
n_best_components = #TODO: decide based on results
models, _, X_pls = pls_da(tf_adata = tf_adata, 
                                      n_components = n_best_components,
                                      control_confounders = True, 
                                      assess = False,
                                      return_components = True,
                                      seed = seed, 
                         enc_X = None, 
                         enc_Y = None)
pls_model = models['pls_model']

In [ ]:
proj_df = pd.DataFrame(X_pls)
proj_df.columns = ['PLS{}'.format(i+1) for i in range(proj_df.shape[1])]


In [ ]:
res = []
for cov_ in ['drug', 'cell_line']:
    cov = tf_adata.obs[cov_].astype(str)
    enc = OneHotEncoder(drop="first", sparse_output=False)
    cov_encoded = enc.fit_transform(cov.values.reshape(-1, 1))

    r2_scores = []
    for pls_idx in trange(proj_df.shape[1]):
        y = proj_df.iloc[:, pls_idx]
        model = LinearRegression().fit(cov_encoded, y)
        r2 = model.score(cov_encoded, y)
        r2_scores.append({"PLS": pls_idx + 1, "R2": r2})

    r2_df = pd.DataFrame(r2_scores)
    r2_df['covariate'] = cov_
    res.append(r2_df)
    
r2_df = pd.concat(res, axis=0, ignore_index=True)


In [ ]:
r2_df.groupby('covariate').R2.mean()

In [ ]:
r2_df.groupby('covariate').R2.median()

In [ ]:
fig, ax = plt.subplots()

pval = stats.mannwhitneyu(r2_df[r2_df.covariate == 'drug'].R2, 
                  r2_df[r2_df.covariate == 'cell_line'].R2).pvalue
sns.boxplot(data = r2_df, x = 'covariate', y = 'R2', ax = ax)

ax.annotate(
    "MWU p-val: {:.2E}".format(pval),
    xy=(0.98, 0.98),
    xycoords="axes fraction",
    ha="right",
    va="top"
)

ax.set_ylabel('R2 (PLS ~ Covariate)')
ax.set_xlabel('Covariate')
ax.set_title('Top 2 PLS')

In [ ]:
proj_df['drug'] = tf_adata.obs.drug.values
proj_df['cell_line'] = tf_adata.obs.cell_line.values
proj_df['condition'] = proj_df.drug.astype(str) + '^' + proj_df.cell_line.astype(str)

In [ ]:
# subset to 20% of the dataset
# n_per_condition = int(np.round(proj_df.drug.value_counts().min() * 0.2))

# # Subsample indices evenly per condition
# sampled_indices = (
#     proj_df.groupby('drug')
#     .sample(n=n_per_condition, random_state=seed)
#     .index
# )

n_per_condition = int(np.round(tf_adata.obs['condition'].value_counts().min() * 0.2))

# Subsample indices evenly per condition
sampled_indices = (
    proj_df.groupby('condition')
    .sample(n=n_per_condition, random_state=seed)
    .index
)

# shuffle
np.random.seed(seed)
sampled_indices=np.random.permutation(sampled_indices)

viz_df = proj_df.loc[sampled_indices, :].copy()

fig, ax = plt.subplots(ncols = 2, figsize = (10, 5))


# DRUG
sns.scatterplot(data = viz_df, x = 'PLS1', y = 'PLS2', hue = 'drug', 
               s = 10, ax = ax[0])
ax[0].legend_.remove()
ax[0].set_title('Drug')

# Cell Line
sns.scatterplot(data = viz_df, x = 'PLS1', y = 'PLS2', hue = 'cell_line', 
               s = 10, ax = ax[1])
ax[1].legend_.remove()
ax[1].set_title('Cell Line')

;

Next, also fit the UMAP with the PLS and drug covariates:

Step 2: Fit a guided UMAP on the PLS components

In [ ]:
%%time
umap_model = umap.UMAP(n_neighbors=15, 
                    n_components=2,
                    metric='euclidean', 
                    target_metric='categorical', 
                    random_state = seed)
umap_model.fit(X_pls,
               tf_adata.obs['drug'].cat.codes.values)
embedding = umap_model.transform(X_pls)

In [ ]:
viz_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'])
viz_df['drug'] = tf_adata.obs.drug.astype(str).values
viz_df['cell_line'] = tf_adata.obs.cell_line.astype(str).values
viz_df = viz_df.loc[sampled_indices, :]

fig, ax = plt.subplots(ncols = 2, figsize = (10, 5))


# DRUG
sns.scatterplot(data = viz_df, x = 'UMAP1', y = 'UMAP2', hue = 'drug', 
               s = 10, ax = ax[0])
ax[0].legend_.remove()
ax[0].set_title('Drug')

# Cell Line
sns.scatterplot(data = viz_df, x = 'UMAP1', y = 'UMAP2', hue = 'cell_line', 
               s = 10, ax = ax[1])
ax[1].legend_.remove()
ax[1].set_title('Cell Line')

;

In [ ]:
for model_type, model in models.items():
    if model is not None:
        joblib.dump(model, os.path.join(data_path, 'processed', author + '_drug_' + model_type + '.joblib'))
joblib.dump(umap_model, os.path.join(data_path, 'processed', author + '_drug_umap.joblib'))

# Use downstream

get the pipeline to project newly predicted values into this embedding

Load the trained models:

In [ ]:
import joblib

pls_model = joblib.load(os.path.join(data_path, 'processed', author + '_drug_pls_model.joblib'))
enc_X = joblib.load(os.path.join(data_path, 'processed', author + '_drug_encoder_x.joblib'))
enc_Y = joblib.load(os.path.join(data_path, 'processed', author + '_drug_encoder_y.joblib'))
umap_model = joblib.load(os.path.join(data_path, 'processed', author + '_drug_umap.joblib'))

X_pred, _ = prepare_input_matrix(tf_adata = tf_adata_predicted, 
                           control_confounders = True, 
                           enc_X = enc_X)
X_pls_pred = pls_model.transform(X_pred)
embedding_predicted = umap_model.transform(X_pls_pred)